# Natural Gas Storage Contract — Pricing Model
    Task 2: Price a commodity storage contract given injection/withdrawal
    dates, volumes, rates, storage capacity, and cost parameters.


# Cash flows considered
      + Revenue   : gas sold at withdrawal price × volume withdrawn
      - Cost      : gas bought at injection price × volume injected
      - Storage   : monthly fee × months gas sits in storage
      - Injection : per-MMBtu fee on every unit injected
      - Withdrawal: per-MMBtu fee on every unit withdrawn
# Steps
      1. Rebuild the price estimator from Task 1
      2. Define the contract_value() pricing function
      3. Validate inputs (capacity, rate, date ordering)
      4. Run test cases and produce a summary report

### Import libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from datetime import datetime

# Price estimate function building

In [2]:
gass=pd.read_csv("Nat_Gas.csv")
gass.columns=gass.columns.str.strip()
gass["Dates"]=pd.to_datetime(gass["Dates"],format="%m/%d/%y")
gass["Prices"]=gass["Prices"].astype(float)
gass=gass[["Dates","Prices"]].sort_values("Dates").reset_index(drop=True)

In [3]:
# Convert dates to a numeric "years since start" for regression
t0=gass["Dates"].min()
gass["t"]=(gass['Dates']-t0).dt.days/365.25

In [4]:
slope,intercept,r_value,p_value,std_err=stats.linregress(gass['t'],gass['Prices'])
gass['trend']= intercept+ slope*gass["t"]
gass['residual']= gass["Prices"] -gass["trend"]

In [8]:
# Monthly seasonal index = mean residual per calendar month
gass["Month"] = gass["Dates"].dt.month
seasonal_idx= gass.groupby('Month')["residual"].mean()
print("Linear trend:")
print(f"  Intercept : ${intercept:.4f}  (price at t=0, Oct 2020)")
print(f"  Slope     : ${slope:.4f} per year")
print(f"  R²        : {r_value**2:.4f}")
print("\nMonthly seasonal index (± vs trend):")

Linear trend:
  Intercept : $10.2910  (price at t=0, Oct 2020)
  Slope     : $0.4684 per year
  R²        : 0.5196

Monthly seasonal index (± vs trend):


In [9]:
def estimate_price(date_str: str) -> float:
    """Return estimated nat-gas price (USD/MMBtu) for any date."""
    date  = pd.to_datetime(date_str)
    t     = (date - t0).days / 365.25
    month = date.month
    return round(intercept + slope * t + seasonal_idx[month], 4)

In [10]:
print(f"Price model ready.  Trend slope: ${slope:.4f}/yr")
print(f"Sample check — Jun 2025: ${estimate_price('2025-06-30'):.2f}  |  "
      f"Dec 2024: ${estimate_price('2024-12-31'):.2f}")

Price model ready.  Trend slope: $0.4684/yr
Sample check — Jun 2025: $11.87  |  Dec 2024: $12.87
